# Filtering

In [ ]:
import json, shutil, time
from collections import Counter
from pathlib import Path

UMLS_META_DIR = Path(r"D:\Project\Radiology_RAG\data\raw_data\umls\umls_2026AA_full\2026AA\META")
OUT = Path(r"D:\Project\Radiology_RAG\data\subtask_data_coverage\filtered_data")
OUT.mkdir(parents=True, exist_ok=True)

## English Filter

In [ ]:
UMLS_COLS, UMLS_ROWS = {}, {}
with open(UMLS_META_DIR/"MRFILES.RRF", encoding="utf-8") as fr:
    for line in fr:
        f = line.split("|")
        name = f[0].removesuffix(".RRF")
        UMLS_COLS[name] = f[2].split(",")
        UMLS_ROWS[name] = int(f[4])

COL = {t: {c: i for i, c in enumerate(cols)} for t, cols in UMLS_COLS.items()}
for t in ("MRCONSO", "MRSTY", "MRDEF", "MRREL", "MRHIER"):
    print(f"{t:<8} {len(UMLS_COLS[t]):>2} cols | {UMLS_ROWS[t]:>12,} rows")

In [ ]:
def eng_file(table):
    """Path of the English-filtered file for a table."""
    return OUT/f"{table}_ENG.RRF"



def filter_rrf(table, keep, every=5_000_000):
    """Stream META/<table>.RRF -> OUT/<table>.RRF, writing each kept line unchanged."""
    src, dst = UMLS_META_DIR/f"{table}.RRF", eng_file(table)
    n_cols = len(UMLS_COLS[table])
    t0, n, kept = time.time(), 0, 0
    with open(src, encoding="utf-8", errors="replace", newline="") as fin, \
        open(dst, "w", encoding="utf-8", newline="") as fout:
        for line in fin:
            n += 1
            f = line.rstrip("\r\n").split("|")
            if len(f) >= n_cols and keep(f):
                fout.write(line)
                kept += 1
            if n % every == 0:
                print(f"{table}: {n:,} read | {kept:,} kept | {time.time()-t0:.0f}s")

    complete = (n == UMLS_ROWS[table])
    print(f"{table:<8} read {n:,} | kept {kept:,} ({kept/n:.1%}) | "
          f"{dst.stat().st_size/1e9:.2f} GB | {time.time()-t0:.0f}s  complete read: {complete}")
    if not complete:
        print(f"  WARNING: expected {UMLS_ROWS[table]:,} rows per MRFILES.RRF")
    return {"read": n, "kept": kept, "complete_read": complete}

stats = {}


In [ ]:
for t in ("MRSTY", "MRDEF", "MRREL", "MRHIER"):
    old, new = OUT/f"{t}.RRF", eng_file(t)
    if old.exists() and not new.exists():
        old.rename(new)
        print(f"renamed {old.name} -> {new.name}")

dup = OUT/"MRCONSO.RRF"
if dup.exists() and eng_file("MRCONSO").exists():
    same = dup.stat().st_size == eng_file("MRCONSO").stat().st_size
    print(f"{dup.name} duplicates {eng_file('MRCONSO').name} (same size: {same}); "
          f"delete it by hand to free {dup.stat().st_size/1e9:.2f} GB")

In [ ]:
# Keep an atom (one name of a concept) if it is English and not suppressed.
# Side effects: collect surviving concepts and the sources that supply English names.
# keep_mask replaces the 3.3M-string keep_cuis set: keep_mask[int("C0040405"[1:])] == 1 means kept.
# If a CUI number is ever >= CUI_CAP, the IndexError tells you to raise CUI_CAP.


CUI_CAP = 10_000_000
C = COL["MRCONSO"]
keep_mask = bytearray(CUI_CAP)
eng_sabs  = Counter()          # SAB -> number of English atoms it contributed

def keep_conso(f):
    if f[C["LAT"]] == "ENG" and f[C["SUPPRESS"]] == "N":
        keep_mask[int(f[C["CUI"]][1:])] = 1
        eng_sabs[f[C["SAB"]]] += 1
        return True
    return False

stats["MRCONSO"] = filter_rrf("MRCONSO", keep_conso, every=2_000_000)
print(f"\nSurviving concepts: {keep_mask.count(1):,} | English sources: {len(eng_sabs)}")
print("Top sources:", ", ".join(f"{s} ({k:,})" for s, k in eng_sabs.most_common(12)))

In [ ]:
C = COL["MRCONSO"]
if "keep_mask" not in globals():
    CUI_CAP = 10_000_000
    keep_mask, eng_sabs = bytearray(CUI_CAP), Counter()
    with open(eng_file("MRCONSO"), encoding="utf-8", errors="replace") as fh:
        for line in fh:
            f = line.split("|")
            keep_mask[int(f[C["CUI"]][1:])] = 1
            eng_sabs[f[C["SAB"]]] += 1
    print(f"rebuilt keep_mask: {keep_mask.count(1):,} concepts")

kept = lambda cui: keep_mask[int(cui[1:])]
S, D, R, H = COL["MRSTY"], COL["MRDEF"], COL["MRREL"], COL["MRHIER"]

stats["MRSTY"] = filter_rrf("MRSTY", lambda f: kept(f[S["CUI"]]))

stats["MRDEF"] = filter_rrf("MRDEF", lambda f:
    kept(f[D["CUI"]]) and f[D["SUPPRESS"]] == "N" and f[D["SAB"]] in eng_sabs)

# MRREL has two CUI columns, so a merge-join can't work here; the mask makes both lookups O(1).
stats["MRREL"] = filter_rrf("MRREL", lambda f:
    kept(f[R["CUI1"]]) and kept(f[R["CUI2"]])
    and f[R["SUPPRESS"]] == "N" and f[R["SAB"]] in eng_sabs)

stats["MRHIER"] = filter_rrf("MRHIER", lambda f:
    kept(f[H["CUI"]]) and f[H["SAB"]] in eng_sabs)

## Relevance Filter

In [ ]:
import re, csv, urllib.request
from itertools import groupby

REF_OUT = Path(r"D:\Project\Radiology_RAG\data\subtask_data_coverage\reference_data")
REF_OUT.mkdir(parents=True, exist_ok=True)

DROP_SAB = {
    # Biology and genetics
    "NCBI":     "NCBI Taxonomy: organism and strain names",
    "HGNC":     "HUGO gene symbols; short symbols (e.g. MET, CAT) collide with ordinary words",
    "OMIM":     "Gene/inherited-disorder catalogue; disease names also exist in SNOMED CT and MeSH",
    "GO":       "Gene Ontology: molecular functions and cell processes",
    # Lab, billing and procedure codes
    "LNC":      "LOINC: laboratory and observation test names",
    "ICD10PCS": "ICD-10-PCS: constructed procedure codes for billing",
    "HCPCS":    "HCPCS: US billing codes for supplies and services",
    "CDT":      "Current Dental Terminology: dental billing codes",
    # Drug products (ingredients stay via MeSH, SNOMED CT, NCI)
    "RXNORM":   "RxNorm: clinical drug product strings",
    "MTHSPL":   "FDA Structured Product Labels",
    "MMSL":     "Multum MediSource Lexicon (commercial drug database)",
    "NDDF":     "First Databank drug file (commercial drug database)",
    "VANDF":    "US Veterans Affairs National Drug File",
    "GS":       "Gold Standard Drug Database (commercial)",
    "DRUGBANK": "DrugBank: drug chemistry and targets",
    "MMX":      "Micromedex (commercial drug database)",
    "MED-RT":   "Medication Reference Terminology: drug classes and mechanisms",
    "ATC":      "WHO Anatomical Therapeutic Chemical classification codes",
    "USP":      "US Pharmacopeia categories",
    "CVX":      "CDC vaccine codes",
}


In [ ]:
M = COL["MRSAB"]
release_sabs = set()
with open(UMLS_META_DIR/"MRSAB.RRF", encoding="utf-8") as fh:
    for line in fh:
        f = line.split("|")
        if f[M["CURVER"]] == "Y":
            release_sabs.add(f[M["RSAB"]])
missing = set(DROP_SAB) - release_sabs
assert not missing, f"not in this release: {missing}"
print(f"{len(DROP_SAB)} sources to drop, all present in the release")

In [ ]:
# Cell 8 — semantic-type policy, built on NLM Semantic Groups (McCray et al., Medinfo 2001).
# Peak RAM ~0 (127 types).
# Rule: drop a type only if none of its concepts can name something seen in, or relevant to,
# the imaged patient (anatomy, finding, procedure, device, substance, pathogen, exposure,
# patient or care setting). When in doubt, keep: a larger reference gives a stricter coverage estimate.
SEMGROUPS_URL = "https://www.nlm.nih.gov/research/umls/knowledge_sources/semantic_network/SemGroups.txt"
SEMGROUPS = REF_OUT/"SemGroups.txt"
if not SEMGROUPS.exists():
    urllib.request.urlretrieve(SEMGROUPS_URL, SEMGROUPS)

TUI_GROUP, TUI_NAME, GROUP_NAME = {}, {}, {}
with open(SEMGROUPS, encoding="utf-8") as fh:
    for line in fh:
        if line.strip():
            grp, grp_name, tui, sty = line.rstrip("\r\n").split("|")
            TUI_GROUP[tui], TUI_NAME[tui], GROUP_NAME[grp] = grp, sty, grp_name

# 1) Whole groups dropped
EXCLUDE_GROUPS = {
    "GENE": "Genes & Molecular Sequences: molecular level, never named as an imaging finding",
    "GEOG": "Geographic Areas: place names",
    "ORGA": "Organizations: institutions and societies",
}

# 2) Exceptions kept inside a dropped group
KEEP_TYPES = {
    "T093": "care settings appear in captions, e.g. 'CXR during 2nd ED visit'",  # Health Care Related Organization (ORGA)
}

# 3) Single types dropped inside groups that are otherwise kept
EXCLUDE_TYPES = {
    # LIVB Living Beings: drop non-pathogen organisms. Kept: T001 Organism, T004 Fungus,
    # T005 Virus, T007 Bacterium (pathogens) and T016, T096-T101 (people and groups).
    "T002": "non-pathogen organism; e.g. peanut the plant (C0030736) goes, peanut the food (C0949399, T168) stays",  # Plant
    "T008": "non-pathogen organism",                                       # Animal
    "T010": "non-pathogen organism",                                       # Vertebrate
    "T011": "non-pathogen organism",                                       # Amphibian
    "T012": "non-pathogen organism",                                       # Bird
    "T013": "non-pathogen organism",                                       # Fish
    "T014": "non-pathogen organism",                                       # Reptile
    "T015": "non-pathogen organism",                                       # Mammal
    "T194": "non-pathogen organism",                                       # Archaeon
    "T204": "mostly non-pathogen organisms; parasites such as Echinococcus (C0322177) are lost, "
            "but their diseases stay as T047",                             # Eukaryote
    # CHEM Chemicals & Drugs
    "T200": "drug product strings ('... 500 MG Oral Tablet'); ingredients stay as T121/T109",  # Clinical Drug
    # CONC Concepts & Ideas (other CONC types are generic but kept; filtering them is a gold-concept step)
    "T089": "laws and regulations",                                        # Regulation or Law
    "T171": "names of languages",                                          # Language
    "T185": "names of classification systems",                             # Classification
    # PROC Procedures
    "T065": "education, not care",                                         # Educational Activity
}

# Guards: the policy must be consistent with SemGroups
for t in {**KEEP_TYPES, **EXCLUDE_TYPES}:
    assert t in TUI_GROUP, f"{t} is not in SemGroups"
for t in KEEP_TYPES:
    assert TUI_GROUP[t] in EXCLUDE_GROUPS, f"{t} is kept but its group is not dropped"
for t in EXCLUDE_TYPES:
    assert TUI_GROUP[t] not in EXCLUDE_GROUPS, f"{t} is redundant: its group is already dropped"

DROP_TUI = frozenset(
    {t for t, g in TUI_GROUP.items() if g in EXCLUDE_GROUPS and t not in KEEP_TYPES}
    | set(EXCLUDE_TYPES)
)

def tui_decision(t):
    """(decision, reason) for one TUI, used in the audit table."""
    g = TUI_GROUP[t]
    if t in EXCLUDE_TYPES:  return "drop", EXCLUDE_TYPES[t]
    if t in KEEP_TYPES:     return "keep", "exception: " + KEEP_TYPES[t]
    if g in EXCLUDE_GROUPS: return "drop", "group: " + EXCLUDE_GROUPS[g]
    return "keep", f"group {g} ({GROUP_NAME[g]}) kept"

print(f"{len(TUI_GROUP)} types in {len(GROUP_NAME)} groups | dropping {len(DROP_TUI)} types:")
for t in sorted(DROP_TUI, key=lambda t: (TUI_GROUP[t], t)):
    print(f"  {TUI_GROUP[t]}  {t}  {TUI_NAME[t]}")

In [ ]:
# Cell 9 — name filters, normalizer and the sorted-block reader. Peak RAM ~0.
MAX_WORDS = 8     # names longer than this almost never occur verbatim in a ~19-word caption;
                  # a parameter to vary in a sensitivity check
MIN_CHARS = 2     # drops single-character names
WS = re.compile(r"\s+")
N_CONSO = len(UMLS_COLS["MRCONSO"])

def norm(s):
    """Shared normalizer for UMLS names AND captions: lowercase, hyphens -> spaces,
    collapse whitespace. 'Ground-glass  opacity' -> 'ground glass opacity'."""
    return WS.sub(" ", s.lower().replace("-", " ")).strip()

def cui_blocks(path):
    """Yield (CUI, [lines]) one CUI at a time. Raises if the file is not sorted by CUI."""
    prev = ""
    with open(path, encoding="utf-8", errors="replace", newline="") as fh:
        for cui, grp in groupby(fh, key=lambda l: l[:l.index("|")]):
            if cui <= prev:
                raise ValueError(f"{path.name} is not sorted by CUI: {cui} after {prev}")
            prev = cui
            yield cui, list(grp)

In [ ]:
# Cell 10 — ONE merge-join pass: MRCONSO_ENG + MRSTY_ENG -> REF_OUT/MRCONSO.RRF, MRSTY.RRF, terms.tsv
# Peak RAM: one CUI block at a time plus 127-key counters, well under 50 MB.
# All-types rule: a concept is dropped by type only if EVERY one of its TUIs is in DROP_TUI.
C, S = COL["MRCONSO"], COL["MRSTY"]
n = Counter()
sab_dropped = Counter()                  # SAB -> names removed
tui_in, tui_dropped = Counter(), Counter()
mrsty_name = {}                          # TUI -> STY as written in this release
t0 = time.time()

conso_blocks = cui_blocks(eng_file("MRCONSO"))
sty_blocks   = cui_blocks(eng_file("MRSTY"))
s_cui, s_lines = next(sty_blocks, (None, None))

with open(REF_OUT/"MRCONSO.RRF", "w", encoding="utf-8", newline="") as out_conso, \
     open(REF_OUT/"MRSTY.RRF",   "w", encoding="utf-8", newline="") as out_sty, \
     open(REF_OUT/"terms.tsv",   "w", encoding="utf-8", newline="") as out_terms:
    out_terms.write("CUI\tterm\tsources\tis_pref\n")

    for cui, c_lines in conso_blocks:
        n["cuis_in"] += 1
        n["atoms_in"] += len(c_lines)
        if n["cuis_in"] % 500_000 == 0:
            print(f"  {n['cuis_in']:,} CUIs | {n['cuis_kept']:,} kept | {time.time()-t0:.0f}s")

        while s_cui is not None and s_cui < cui:          # both sorted: MRSTY only moves forward
            s_cui, s_lines = next(sty_blocks, (None, None))
        if s_cui != cui:
            n["drop_no_sty"] += 1
            continue

        tuis = set()
        for l in s_lines:
            f = l.split("|")
            tuis.add(f[S["TUI"]])
            mrsty_name.setdefault(f[S["TUI"]], f[S["STY"]])
        tui_in.update(tuis)
        if tuis <= DROP_TUI:
            n["drop_type"] += 1
            tui_dropped.update(tuis)
            continue

        kept_lines, terms = [], {}                         # term -> [set(SABs), is_pref]
        for line in c_lines:
            f = line.split("|")
            if len(f) < N_CONSO:
                n["drop_malformed"] += 1
                continue
            sab, s = f[C["SAB"]], f[C["STR"]]
            if sab in DROP_SAB:
                sab_dropped[sab] += 1
                continue
            if len(s) < MIN_CHARS or len(s.split()) > MAX_WORDS:
                n["drop_length"] += 1
                continue
            kept_lines.append(line)
            entry = terms.setdefault(norm(s), [set(), False])
            entry[0].add(sab)
            if f[C["TS"]] == "P" and f[C["STT"]] == "PF" and f[C["ISPREF"]] == "Y":
                entry[1] = True

        if not kept_lines:
            n["drop_no_names_left"] += 1                   # every name came from dropped sources or was too long
            continue

        out_conso.writelines(kept_lines)
        out_sty.writelines(s_lines)
        for term, (sabs, pref) in terms.items():
            out_terms.write(f"{cui}\t{term}\t{','.join(sorted(sabs))}\t{pref}\n")
        n["cuis_kept"]  += 1
        n["atoms_kept"] += len(kept_lines)
        n["sty_kept"]   += len(s_lines)
        n["terms"]      += len(terms)

# Guard: SemGroups and this release must describe the same 127 types
assert set(mrsty_name) == set(TUI_GROUP), f"TUI mismatch: {set(mrsty_name) ^ set(TUI_GROUP)}"
renamed = {t: (TUI_NAME[t], mrsty_name[t]) for t in TUI_GROUP if TUI_NAME[t] != mrsty_name[t]}
if renamed:
    print("NOTE: type names differ between SemGroups and MRSTY (TUIs still match):", renamed)

print(f"\nCUIs   {n['cuis_in']:,} -> {n['cuis_kept']:,}")
print(f"atoms  {n['atoms_in']:,} -> {n['atoms_kept']:,} (expect atoms_in = 9,112,359)")
print(f"terms  {n['terms']:,} unique (CUI, name) | MRSTY rows {n['sty_kept']:,} | {time.time()-t0:.0f}s")
print("drops:", {k: v for k, v in n.items() if k.startswith("drop")})

In [ ]:
# Cell 11 — audit tables for the thesis appendix. Peak RAM ~0.
with open(REF_OUT/"sty_decisions.csv", "w", encoding="utf-8", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["group", "group_name", "TUI", "semantic_type", "decision", "reason",
                "concepts_with_type", "concepts_dropped_by_type_rule"])
    for t in sorted(TUI_GROUP, key=lambda t: (TUI_GROUP[t], t)):
        dec, why = tui_decision(t)
        g = TUI_GROUP[t]
        w.writerow([g, GROUP_NAME[g], t, TUI_NAME[t], dec, why, tui_in[t], tui_dropped[t]])

with open(REF_OUT/"sab_decisions.csv", "w", encoding="utf-8", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["SAB", "reason", "english_names_dropped"])
    for sab, why in sorted(DROP_SAB.items(), key=lambda kv: -sab_dropped[kv[0]]):
        w.writerow([sab, why, sab_dropped[sab]])

print("Largest type drops:")
for t, k in tui_dropped.most_common(6):
    print(f"  {t} {TUI_NAME[t]:<40} {k:>9,}")
print(f"Concepts that lost every name to the source/length filters: {n['drop_no_names_left']:,}")

In [ ]:
# Cell 12 — sanity check by CUI (looked up in your files). Peak RAM ~0 (streams terms.tsv).
must_have = {
    "C0021708": "intensive care unit",
    "C0026926": "Mycobacterium tuberculosis (T007, pathogen kept)",
    "C0240984": "scuba diving (T056, barotrauma)",
    "C0009133": "coal mining (T090, occupational exposure)",
    "C0004754": "barium sulfate (contrast agent)",
    "C0949399": "peanut as food (aspiration foreign body)",
    "C0021270": "infant (T100)",
}
# must_have_terms = {"pneumothorax", "pleural effusion", "mediastinum", "ground-glass opacity", "atelectasis"}

must_have["C5539411"] = "ground glass opacity (finding)"
must_have_terms = {"pneumothorax", "pleural effusion", "mediastinum", "ground glass opacity", "atelectasis"}

must_not = {
    "C0376571": "BRCA1 gene (T028)",
    "C0322177": "Echinococcus granulosus, the organism (T204)",
    "C0030736": "peanut, the plant (T002)",
}
seen_cuis, seen_terms = set(), set()
with open(REF_OUT/"terms.tsv", encoding="utf-8") as fh:
    next(fh)
    for line in fh:
        cui, term = line.split("\t", 2)[:2]
        if cui in must_have or cui in must_not:
            seen_cuis.add(cui)
        if term in must_have_terms:
            seen_terms.add(term)

for c, label in must_have.items(): print(f"{'OK     ' if c in seen_cuis else 'MISSING'} {c} {label}")
for t in sorted(must_have_terms):  print(f"{'OK     ' if t in seen_terms else 'MISSING'} {t}")
for c, label in must_not.items():  print(f"{'LEAK   ' if c in seen_cuis else 'OK     '} {c} {label} (should be absent)")

# Coverage Cells

## Attempt 1

In [ ]:
# Cell 13 — coverage config. Peak RAM ~0.
import gc
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS   # scikit-learn's 318-word list

In [ ]:
COV_OUT = Path(r"D:\Project\Radiology_RAG\data\subtask_data_coverage\coverage_results")
COV_OUT.mkdir(parents=True, exist_ok=True)

MLB_CHEST = Path(r"D:\Project\Radiology_RAG\data\MedLayBench-V\chest")
DATASETS = {
    "mlbv_chest_validation": MLB_CHEST/"validation_chest_text_final.parquet",
    "mlbv_chest_test":       MLB_CHEST/"test_chest_text_final.parquet",
}
TEXT_COL = "expert_caption"


# Semantic Groups counted as "clinical". CONC (generic ideas like "left", "image"),
# OBJC, ACTI and OCCU are still reported, but in their own rows.
CLINICAL_GROUPS = {"ANAT", "DISO", "PROC", "DEVI", "CHEM", "PHYS", "PHEN", "LIVB"}

TOKEN  = re.compile(r"[a-z0-9]+")

def key(s):
    """Match key applied to BOTH captions and UMLS names: norm(), then alphanumeric tokens.
    'Ground-glass opacity, bilateral.' -> 'ground glass opacity bilateral'"""
    return " ".join(TOKEN.findall(norm(s)))

In [ ]:
# Cell 15 — tokenize captions, collect every n-gram up to MAX_WORDS.
# Peak RAM ~100 MB (~2 x 2.8k captions x ~140 n-grams).
captions = {}
for ds, path in DATASETS.items():
    df = pd.read_parquet(path, columns=[TEXT_COL])
    captions[ds] = [key(t).split() if isinstance(t, str) else [] for t in df[TEXT_COL]]
    del df
    print(f"{ds:<24} {len(captions[ds]):,} captions")

grams = set()
for toks_list in captions.values():
    for toks in toks_list:
        for n in range(1, MAX_WORDS + 1):
            for i in range(len(toks) - n + 1):
                grams.add(" ".join(toks[i:i+n]))
print(f"distinct caption n-grams (1-{MAX_WORDS} words): {len(grams):,}")

In [ ]:
# Cell 16 — lexicon: stream terms.tsv, keep only names that occur in some caption.
# Peak RAM ~100 MB (the n-gram set) + small LEX (a few 10k keys). ~1 min.
LEX = {}                                   # key -> set of candidate CUIs
n_terms = n_stop = 0
with open(REF_OUT/"terms.tsv", encoding="utf-8") as fh:
    next(fh)
    for line in fh:
        n_terms += 1
        cui, term = line.split("\t", 2)[:2]
        k = key(term)
        if k not in grams:
            continue
        if k in ENGLISH_STOP_WORDS or k.replace(" ", "").isdigit():
            n_stop += 1                    # "the", "was", "10" are UMLS names but not concepts here
            continue
        LEX.setdefault(k, set()).add(cui)

del grams; gc.collect()
amb = sum(len(v) > 1 for v in LEX.values())
print(f"terms read {n_terms:,} | lexicon keys {len(LEX):,} | ambiguous keys {amb:,} "
      f"({amb/len(LEX):.1%}) | skipped stopword/number names {n_stop:,}")

In [ ]:
# Cell 17 — greedy longest-match extraction; one mentions file per dataset.
# Peak RAM: mentions list (~10 per caption), a few MB.
def extract(toks):
    """Left to right; at each position take the longest key in LEX (<= MAX_WORDS words)."""
    i, out = 0, []
    while i < len(toks):
        for n in range(min(MAX_WORDS, len(toks) - i), 0, -1):
            k = " ".join(toks[i:i+n])
            if k in LEX:
                out.append((i, n, k))
                i += n
                break
        else:
            i += 1
    return out

MENTIONS = {}                              # ds -> [(row, start, n_words, key, cuis_tuple)]
for ds, toks_list in captions.items():
    ments = []
    with open(COV_OUT/f"{ds}_mentions.tsv", "w", encoding="utf-8", newline="") as fh:
        fh.write("row\tstart\tn_words\tkey\tcuis\n")
        for row, toks in enumerate(toks_list):
            for start, n, k in extract(toks):
                cuis = tuple(sorted(LEX[k]))
                ments.append((row, start, n, k, cuis))
                fh.write(f"{row}\t{start}\t{n}\t{k}\t{';'.join(cuis)}\n")
    MENTIONS[ds] = ments
    rows_with = len({m[0] for m in ments})
    print(f"{ds:<24} {len(ments):,} mentions | {len(ments)/len(toks_list):.1f} per caption | "
          f"captions with >=1 match {rows_with:,}/{len(toks_list):,}")

In [ ]:
# Cell 18 — semantic groups of the matched CUIs only. Peak RAM: a few MB (streams MRSTY).
matched = {c for ments in MENTIONS.values() for m in ments for c in m[4]}
S = COL["MRSTY"]
CUI_GROUPS = {}                            # cui -> {group, ...}
with open(REF_OUT/"MRSTY.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        cui = line[:line.index("|")]
        if cui in matched:
            CUI_GROUPS.setdefault(cui, set()).add(TUI_GROUP[line.split("|")[S["TUI"]]])
print(f"matched concepts {len(matched):,} | with groups {len(CUI_GROUPS):,}")

In [ ]:
# Cell 19 — token coverage: share of caption content words that fall inside a UMLS match.
# Content word = not a stopword, not a pure number. Peak RAM ~0.
def is_content(t):
    return t not in ENGLISH_STOP_WORDS and not t.isdigit()

def is_clinical(cuis):
    return any(CUI_GROUPS.get(c, set()) & CLINICAL_GROUPS for c in cuis)

rows = []
for ds, toks_list in captions.items():
    by_row = {}
    for row, start, n, k, cuis in MENTIONS[ds]:
        by_row.setdefault(row, []).append((start, n, cuis))
    tot = cov_all = cov_clin = 0
    D, D_clin, D_strict = set(), set(), set()
    for row, toks in enumerate(toks_list):
        in_any, in_clin = set(), set()
        for start, n, cuis in by_row.get(row, []):
            span = range(start, start + n)
            in_any.update(span)
            D.update(cuis)
            if len(cuis) == 1:
                D_strict.add(cuis[0])
            if is_clinical(cuis):
                in_clin.update(span)
                D_clin.update(c for c in cuis if CUI_GROUPS.get(c, set()) & CLINICAL_GROUPS)
        for i, t in enumerate(toks):
            if is_content(t):
                tot += 1
                cov_all  += i in in_any
                cov_clin += i in in_clin
    rows.append({"dataset": ds, "captions": len(toks_list), "content_tokens": tot,
                 "token_cov_any_match": cov_all / tot,
                 "token_cov_clinical_match": cov_clin / tot,
                 "concepts_D": len(D), "concepts_D_clinical": len(D_clin),
                 "concepts_D_unambiguous": len(D_strict)})
cov = pd.DataFrame(rows)
cov.to_csv(COV_OUT/"coverage_summary.csv", index=False)
pd.options.display.float_format = "{:.3f}".format
cov

In [ ]:
# Cell 20 — the concept set D by semantic group (validation split), and D saved for the KB step.
# Peak RAM ~0.
DS = "mlbv_chest_validation"
freq = Counter(c for m in MENTIONS[DS] for c in m[4])

with open(COV_OUT/f"{DS}_concepts_D.tsv", "w", encoding="utf-8", newline="") as fh:
    fh.write("CUI\tmentions\tgroups\tclinical\n")
    for c, v in freq.most_common():
        g = CUI_GROUPS.get(c, set())
        fh.write(f"{c}\t{v}\t{','.join(sorted(g))}\t{bool(g & CLINICAL_GROUPS)}\n")

by_group = []
for g in sorted(GROUP_NAME):
    Dg = [c for c in freq if g in CUI_GROUPS.get(c, set())]
    if Dg:
        by_group.append({"group": g, "name": GROUP_NAME[g], "clinical": g in CLINICAL_GROUPS,
                         "concepts": len(Dg), "mentions": sum(freq[c] for c in Dg)})
grp = pd.DataFrame(by_group).sort_values("concepts", ascending=False)
grp.to_csv(COV_OUT/f"{DS}_D_by_group.csv", index=False)
grp

In [ ]:
# Cell 21 — gaps: the most frequent content words that no UMLS name covered.
# These show what the reference misses (abbreviations, misspellings, rare terms). Peak RAM ~0.
toks_list = captions[DS]
by_row = {}
for row, start, n, k, cuis in MENTIONS[DS]:
    by_row.setdefault(row, set()).update(range(start, start + n))

gaps = Counter()
for row, toks in enumerate(toks_list):
    covered = by_row.get(row, set())
    for i, t in enumerate(toks):
        if is_content(t) and i not in covered:
            gaps[t] += 1

top = pd.DataFrame(gaps.most_common(200), columns=["token", "count"])
top.to_csv(COV_OUT/f"{DS}_uncovered_tokens.csv", index=False)
top.head(40)

##  Coverage Cells (Attempt 2)

In [ ]:
# Cell 13 — coverage config. Peak RAM ~0.
# Uses ONLY this subtask's data: reference_data, MRXNS_ENG (same 2026AA release) and the MedLayBench-V captions.
import gc
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS   # scikit-learn's 318-word list

COV_OUT = Path(r"D:\Project\Radiology_RAG\data\subtask_data_coverage\coverage_results")
COV_OUT.mkdir(parents=True, exist_ok=True)
SUFFIX = "_norm"                           # exact-match results from the earlier run keep their names

MLB_CHEST = Path(r"D:\Project\Radiology_RAG\data\MedLayBench-V\chest")
DATASETS = {
    "mlbv_chest_validation": MLB_CHEST/"validation_chest_text_final.parquet",
    "mlbv_chest_test":       MLB_CHEST/"test_chest_text_final.parquet",
}
TEXT_COL = "expert_caption"

# Semantic Groups counted as "clinical". CONC, OBJC, ACTI and OCCU are still reported, in their own rows.
CLINICAL_GROUPS = {"ANAT", "DISO", "PROC", "DEVI", "CHEM", "PHYS", "PHEN", "LIVB"}

# Words NLM's normalizer drops from names, measured by comparing 396,627 sampled names
# with their MRXNS_ENG normalized strings. "s" is the possessive left after removing the apostrophe.
NORM_STOP = frozenset({"of", "to", "and", "with", "by", "in", "on", "the", "nos", "for", "s"})

TOKEN = re.compile(r"[a-z0-9]+")

def key(s):
    """Tokenize text: norm() (lowercase, hyphens -> spaces), then alphanumeric tokens."""
    return " ".join(TOKEN.findall(norm(s)))

def bad_key(k):
    """Keys that are not concepts here: English stopwords and pure numbers."""
    return k in ENGLISH_STOP_WORDS or k.replace(" ", "").isdigit()

In [ ]:
# Cell 14 — tokenize captions; collect the caption vocabulary. Peak RAM ~20 MB.
captions = {}
for ds, path in DATASETS.items():
    df = pd.read_parquet(path, columns=[TEXT_COL])
    captions[ds] = [key(t).split() if isinstance(t, str) else [] for t in df[TEXT_COL]]
    del df
    print(f"{ds:<24} {len(captions[ds]):,} captions")

VOCAB = {t for tl in captions.values() for toks in tl for t in toks}
print(f"distinct caption words: {len(VOCAB):,}")

In [ ]:
# Cell 15 — uninflection table for caption words. Two streaming passes (~535 MB + ~1.1 GB), ~2 min.
# Peak RAM: a few MB (dicts keyed by caption words only).
def plural_candidates(w):
    """English and Latin plural -> singular candidates. Used only as a fallback, and a candidate
    is accepted only if it occurs as a word in NLM's normalized strings."""
    if len(w) < 4:
        return []
    c = []
    if w.endswith("ies"):  c.append(w[:-3] + "y")                  # arteries -> artery
    if w.endswith("ices"): c += [w[:-4] + "ex", w[:-4] + "ix"]     # apices -> apex
    if w.endswith("ses"):  c.append(w[:-3] + "sis")                # metastases -> metastasis
    if w.endswith("es"):   c.append(w[:-2])                        # masses -> mass
    if w.endswith("s") and not w.endswith("ss"): c.append(w[:-1])  # consolidations -> consolidation
    if w.endswith("ae"):   c.append(w[:-1])                        # vertebrae -> vertebra
    if w.endswith("i"):    c.append(w[:-1] + "us")                 # bronchi -> bronchus
    return c

# Pass A: SUIs of single-word kept names that are also caption words ("Opacities" -> S...)
C = COL["MRCONSO"]
sui_word = {}
with open(REF_OUT/"MRCONSO.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.split("|")
        k = key(f[C["STR"]])
        if k in VOCAB:                     # VOCAB holds single words, so k is a single word
            sui_word[f[C["SUI"]]] = k

# Pass B: NLM's normalized form of those names, and which candidate words NLM's vocabulary contains
X = COL["MRXNS_ENG"]
CAND = VOCAB | {c for w in VOCAB for c in plural_candidates(w)}
base, in_vocab = {}, set()
t0 = time.time()
with open(UMLS_META_DIR/"MRXNS_ENG.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.split("|")
        nstr = f[X["NSTR"]]
        w = sui_word.get(f[X["SUI"]])
        if w is not None and " " not in nstr:
            base.setdefault(w, set()).add(nstr)
        for t in nstr.split():
            if t in CAND:
                in_vocab.add(t)
print(f"MRXNS_ENG pass: {time.time()-t0:.0f}s")

def uninflect(w):
    """(base form, source). Priority: NLM's own normal form of that exact name > word already
    in NLM's vocabulary > plural rule whose result is in NLM's vocabulary > unchanged."""
    b = base.get(w)
    if b:
        return (w, "unchanged") if w in b else (min(b), "umls_name")
    if w in in_vocab:
        return w, "unchanged"
    for c in plural_candidates(w):
        if c in in_vocab:
            return c, "plural_rule"
    return w, "unchanged"

LEMMA, src = {}, Counter()
with open(COV_OUT/f"uninflection_map{SUFFIX}.csv", "w", encoding="utf-8", newline="") as fh:
    w_ = csv.writer(fh); w_.writerow(["caption_word", "base", "source"])
    for w in sorted(VOCAB):
        b, s = uninflect(w)
        LEMMA[w] = b
        src[s] += 1
        if s != "unchanged":
            w_.writerow([w, b, s])
del sui_word, base, CAND; gc.collect()

print("sources:", dict(src))
for w in ["consolidations", "opacities", "bronchi", "vertebrae", "metastases", "nodules", "effusions", "ribs"]:
    if w in LEMMA: print(f"  {w:<15} -> {LEMMA[w]}")

In [ ]:
# Cell 16 — normalized caption n-grams. Peak RAM ~100 MB (the n-gram set).
def nkey(toks):
    """NLM-style normalized key of a token span: drop NORM_STOP, uninflect, sort the words."""
    return " ".join(sorted(LEMMA.get(t, t) for t in toks if t not in NORM_STOP))

def spans(toks):
    """Spans of 1..MAX_WORDS tokens that start and end on a non-stopword token."""
    for i in range(len(toks)):
        if toks[i] in NORM_STOP:
            continue
        for n in range(1, min(MAX_WORDS, len(toks) - i) + 1):
            if toks[i + n - 1] not in NORM_STOP:
                yield i, n

NGRAMS = set()
for tl in captions.values():
    for toks in tl:
        for i, n in spans(toks):
            NGRAMS.add(nkey(toks[i:i+n]))
print(f"distinct normalized caption n-grams: {len(NGRAMS):,}")

In [ ]:
# Cell 17 — lexicon from MRXNS_ENG, restricted to names kept in reference_data. Two streaming passes, ~2 min.
# Peak RAM: NGRAMS (~100 MB) + candidate rows (tens of MB).
# A (CUI, SUI) pair counts only if that exact name survived our filters in reference_data/MRCONSO.RRF,
# so the source, type and length filters still apply.
cand = {}                                  # (CUI, SUI) -> {normalized key}
with open(UMLS_META_DIR/"MRXNS_ENG.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.split("|")
        k = f[X["NSTR"]]
        if k in NGRAMS and not bad_key(k):
            cand.setdefault((f[X["CUI"]], f[X["SUI"]]), set()).add(k)
del NGRAMS; gc.collect()

LEX = {}                                   # normalized key -> {CUI}
with open(REF_OUT/"MRCONSO.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.split("|")
        ks = cand.get((f[C["CUI"]], f[C["SUI"]]))
        if ks:
            for k in ks:
                LEX.setdefault(k, set()).add(f[C["CUI"]])
del cand; gc.collect()

amb = sum(len(v) > 1 for v in LEX.values())
print(f"lexicon keys {len(LEX):,} | ambiguous {amb:,} ({amb/len(LEX):.1%})")
for test in (["consolidations"], ["ground", "glass", "opacities"], ["fracture", "of", "the", "rib"]):
    k = nkey(test)
    print(f"  {' '.join(test):<28} -> '{k}' {'FOUND' if k in LEX else 'not found'} {sorted(LEX.get(k, []))[:3]}")

In [ ]:
# Cell 18 — greedy longest-match extraction; one mentions file per dataset. Peak RAM: a few MB.
def extract(toks):
    """Left to right; at each position take the longest span whose normalized key is in LEX."""
    i, out = 0, []
    while i < len(toks):
        hit = None
        if toks[i] not in NORM_STOP:
            for n in range(min(MAX_WORDS, len(toks) - i), 0, -1):
                if toks[i + n - 1] in NORM_STOP:
                    continue
                k = nkey(toks[i:i+n])
                if k in LEX:
                    hit = (i, n, k)
                    break
        if hit:
            out.append(hit)
            i += hit[1]
        else:
            i += 1
    return out

MENTIONS = {}                              # ds -> [(row, start, n_words, key, cuis_tuple)]
for ds, toks_list in captions.items():
    ments = []
    with open(COV_OUT/f"{ds}_mentions{SUFFIX}.tsv", "w", encoding="utf-8", newline="") as fh:
        fh.write("row\tstart\tn_words\tspan\tkey\tcuis\n")
        for row, toks in enumerate(toks_list):
            for start, n, k in extract(toks):
                cuis = tuple(sorted(LEX[k]))
                ments.append((row, start, n, k, cuis))
                fh.write(f"{row}\t{start}\t{n}\t{' '.join(toks[start:start+n])}\t{k}\t{';'.join(cuis)}\n")
    MENTIONS[ds] = ments
    rows_with = len({m[0] for m in ments})
    print(f"{ds:<24} {len(ments):,} mentions | {len(ments)/len(toks_list):.1f} per caption | "
          f"captions with >=1 match {rows_with:,}/{len(toks_list):,}")

In [ ]:
# Cell 19 — semantic groups of the matched CUIs only. Peak RAM: a few MB (streams MRSTY).
matched = {c for ments in MENTIONS.values() for m in ments for c in m[4]}
S = COL["MRSTY"]
CUI_GROUPS = {}
with open(REF_OUT/"MRSTY.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        cui = line[:line.index("|")]
        if cui in matched:
            CUI_GROUPS.setdefault(cui, set()).add(TUI_GROUP[line.split("|")[S["TUI"]]])
print(f"matched concepts {len(matched):,} | with groups {len(CUI_GROUPS):,}")

In [ ]:
# Cell 20 — token coverage summary. Peak RAM ~0.
def is_content(t):
    return t not in ENGLISH_STOP_WORDS and not t.isdigit()

def clinical_cuis(cuis):
    return [c for c in cuis if CUI_GROUPS.get(c, set()) & CLINICAL_GROUPS]

rows = []
for ds, toks_list in captions.items():
    by_row = {}
    for row, start, n, k, cuis in MENTIONS[ds]:
        by_row.setdefault(row, []).append((start, n, cuis))
    tot = cov_all = cov_clin = 0
    D, D_clin, D_strict = set(), set(), set()
    for row, toks in enumerate(toks_list):
        in_any, in_clin = set(), set()
        for start, n, cuis in by_row.get(row, []):
            span = range(start, start + n)
            in_any.update(span)
            D.update(cuis)
            if len(cuis) == 1:
                D_strict.add(cuis[0])
            clin = clinical_cuis(cuis)
            if clin:
                in_clin.update(span)
                D_clin.update(clin)
        for i, t in enumerate(toks):
            if is_content(t):
                tot += 1
                cov_all  += i in in_any
                cov_clin += i in in_clin
    rows.append({"dataset": ds, "captions": len(toks_list), "content_tokens": tot,
                 "token_cov_any_match": cov_all / tot,
                 "token_cov_clinical_match": cov_clin / tot,
                 "concepts_D": len(D), "concepts_D_clinical": len(D_clin),
                 "concepts_D_unambiguous": len(D_strict)})
cov = pd.DataFrame(rows)
cov.to_csv(COV_OUT/f"coverage_summary{SUFFIX}.csv", index=False)
pd.options.display.float_format = "{:.3f}".format
cov

In [ ]:
# Cell 21 — concept set D by semantic group (validation), and D saved. Peak RAM ~0.
DS = "mlbv_chest_validation"
freq = Counter(c for m in MENTIONS[DS] for c in m[4])

with open(COV_OUT/f"{DS}_concepts_D{SUFFIX}.tsv", "w", encoding="utf-8", newline="") as fh:
    fh.write("CUI\tmentions\tgroups\tclinical\n")
    for c, v in freq.most_common():
        g = CUI_GROUPS.get(c, set())
        fh.write(f"{c}\t{v}\t{','.join(sorted(g))}\t{bool(g & CLINICAL_GROUPS)}\n")

by_group = []
for g in sorted(GROUP_NAME):
    Dg = [c for c in freq if g in CUI_GROUPS.get(c, set())]
    if Dg:
        by_group.append({"group": g, "name": GROUP_NAME[g], "clinical": g in CLINICAL_GROUPS,
                         "concepts": len(Dg), "mentions": sum(freq[c] for c in Dg)})
grp = pd.DataFrame(by_group).sort_values("concepts", ascending=False)
grp.to_csv(COV_OUT/f"{DS}_D_by_group{SUFFIX}.csv", index=False)
grp

In [ ]:
# Cell 22 — most frequent uncovered content words (validation). Peak RAM ~0.
by_row = {}
for row, start, n, k, cuis in MENTIONS[DS]:
    by_row.setdefault(row, set()).update(range(start, start + n))

gaps = Counter()
for row, toks in enumerate(captions[DS]):
    covered = by_row.get(row, set())
    for i, t in enumerate(toks):
        if is_content(t) and i not in covered:
            gaps[t] += 1

top = pd.DataFrame(gaps.most_common(200), columns=["token", "count"])
top.to_csv(COV_OUT/f"{DS}_uncovered_tokens{SUFFIX}.csv", index=False)
top.head(40)

In [ ]:
# Cell 23 — exact vs normalized matching, side by side (for the methods chapter). Peak RAM ~0.
cmp = pd.concat([
    pd.read_csv(COV_OUT/"coverage_summary.csv").assign(matching="exact"),
    pd.read_csv(COV_OUT/f"coverage_summary{SUFFIX}.csv").assign(matching="nlm_normalized"),
])
cmp[["dataset", "matching", "token_cov_any_match", "token_cov_clinical_match",
     "concepts_D", "concepts_D_clinical", "concepts_D_unambiguous"]]